# Parcial para la casa · Los dos Hermite
## Polinomios osculantes, cuadratura de Gauss–Hermite y la molécula de HCl

**Fuente de los datos.** NIST Chemistry WebBook, SRD 69, página *Hydrogen chloride*, sección *Constants of diatomic molecules* (compilación de K. P. Huber y G. Herzberg, 1979), estado fundamental X ¹Σ⁺ de H³⁵Cl:
https://webbook.nist.gov/cgi/cbook.cgi?ID=C7647010&Mask=1000

Masas atómicas: NIST, *Atomic Weights and Isotopic Compositions*: https://physics.nist.gov/cgi-bin/Compositions/stand_alone.pl

**Fecha de consulta:** 15 de octubre de 2026.
**Licencia / uso:** datos de referencia estándar de NIST (U.S. Department of Commerce). Se usan con fines académicos citando la fuente; ver las condiciones de uso en el sitio de NIST.

La celda 1 intenta leer la página en línea y comprobar que los valores coinciden. Si la URL falla, el cuaderno usa la **copia congelada** que viene embebida (los mismos números, copiados textualmente el 15 oct 2026). Corre las celdas **en orden**.

### Celda 1 · Datos reales (en línea + copia congelada)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import factorial, sqrt, pi, exp
import urllib.request

URL_NIST = "https://webbook.nist.gov/cgi/cbook.cgi?ID=C7647010&Mask=1000"

# Copia congelada (textual, NIST WebBook, 15 oct 2026). Unidades: cm^-1, salvo r_e en angstrom.
DATOS_CONGELADOS = {
    "omega_e":    "2990.9463",   # ωe
    "omega_e_xe": "52.8186",     # ωe·xe
    "omega_e_ye": "0.22437",     # ωe·ye
    "omega_e_ze": "-0.01218",    # ωe·ze (nota [27] de la tabla)
    "r_e":        "1.274552",    # distancia de equilibrio (Å)
}
# Masas atómicas (u), NIST: 1H = 1.007 825 032 23(9), 35Cl = 34.968 852 682(37)
m_H, m_Cl = 1.00782503223, 34.968852682

origen = "copia congelada"
try:
    req = urllib.request.Request(URL_NIST, headers={"User-Agent": "Mozilla/5.0"})
    html = urllib.request.urlopen(req, timeout=15).read().decode("utf-8", "ignore")
    faltan = [k for k, v in DATOS_CONGELADOS.items() if v not in html]
    if not faltan:
        origen = "NIST en línea (los 5 valores aparecen textualmente en la página)"
    else:
        print("Ojo: en la página en línea no encontré", faltan, "-> uso la copia congelada")
except Exception as e:
    print("No pude leer NIST en línea (", type(e).__name__, ") -> uso la copia congelada")

we, wexe, weye, weze, re_A = (float(DATOS_CONGELADOS[k]) for k in
                              ["omega_e", "omega_e_xe", "omega_e_ye", "omega_e_ze", "r_e"])
print("Origen de los datos:", origen)
print(f"ωe = {we} cm⁻¹,  ωe·xe = {wexe} cm⁻¹,  ωe·ye = {weye} cm⁻¹,  ωe·ze = {weze} cm⁻¹,  re = {re_A} Å")

### Celda 2 · Unidades del oscilador (todo queda limpio)

Usamos la coordenada adimensional x = (r − rₑ)/ℓ con ℓ = √(ħ/(μω)), y medimos la energía en unidades de ħω (es decir, en unidades de ωe). Así:

- potencial armónico: V_arm(x) = x²/2, estado base |ψ₀(x)|² = e^{−x²}/√π;
- potencial de Morse: V_M(x) = D(1 − e^{−a x})² con **D = ωe/(4 ωe·xe)** y **a = 1/√(2D)** (así Morse y el armónico tienen la misma curvatura en x = 0).

In [ ]:
D = we / (4 * wexe)          # profundidad del pozo de Morse, en unidades de ħω
a = 1 / np.sqrt(2 * D)       # parámetro de ancho (adimensional)

# Solo para ubicarte en el mundo físico:
hbar, c, uma, eV = 1.054571817e-34, 2.99792458e10, 1.66053906660e-27, 1.602176634e-19
mu = m_H * m_Cl / (m_H + m_Cl)                 # masa reducida (u)
omega = 2 * np.pi * c * we                     # rad/s
ell = np.sqrt(hbar / (mu * uma * omega)) * 1e10  # Å
print(f"D = {D:.4f} ħω   (De ≈ {D*we:.0f} cm⁻¹ ≈ {D*we*2*np.pi*hbar*c/eV:.2f} eV: estimación del modelo de Morse, no es un dato medido)")
print(f"a = {a:.6f}   (en unidades físicas: β = a/ℓ = {a/ell:.4f} Å⁻¹)")
print(f"μ = {mu:.6f} u,   ℓ = {ell:.5f} Å  ->  la vibración del punto cero mide ~{100*ell/re_A:.1f} % de la longitud de enlace re = {re_A} Å")

### Celda 3 · Completa la función (regla de Gauss–Hermite)

La regla con n nodos aproxima ⟨g⟩ = (1/√π)∫ e^{−x²} g(x) dx ≈ (1/√π) Σⱼ wⱼ g(xⱼ), donde los xⱼ son las raíces de Hₙ y wⱼ = ∫ e^{−x²} Lⱼ(x)² dx (Parte I). `np.polynomial.hermite.hermgauss(n)` te da los xⱼ y los wⱼ. Completa **una línea**.

In [ ]:
def valor_esperado_GH(g, n):
    x, w = np.polynomial.hermite.hermgauss(n)   # nodos = raíces de H_n, pesos w_j
    # COMPLETA: devuelve (1/√π) Σ_j w_j g(x_j)
    return None                                  # <-- CAMBIA ESTO

In [ ]:
# Prueba automática: ⟨x²⟩ en el estado base vale 1/2. Si tu función falla, se usa la de respaldo.
def _respaldo(g, n):
    x, w = np.polynomial.hermite.hermgauss(n)
    return np.sum(w * g(x)) / np.sqrt(np.pi)
try:
    ok = abs(valor_esperado_GH(lambda x: x**2, 3) - 0.5) < 1e-12
except Exception:
    ok = False
if ok:
    print("¡Tu función pasa la prueba! ⟨x²⟩ =", valor_esperado_GH(lambda x: x**2, 3))
else:
    print("Tu función aún no pasa la prueba; uso la de respaldo para seguir.")
    valor_esperado_GH = _respaldo

### Celda 4 · El puente: los pesos salen del osculante

Construimos los Lⱼ en los nodos de Gauss–Hermite y calculamos con una regla muy fina (60 nodos, exacta para estos polinomios) los dos tipos de "pesos" que aparecen al integrar el osculante:
- peso de los valores: ∫ e^{−x²} Lⱼ(x)² dx,
- peso de las derivadas: ∫ e^{−x²} (x − xⱼ) Lⱼ(x)² dx (la Parte I dice que vale **0**).

In [ ]:
n = 3                                            # <-- CAMBIA ESTO (prueba 2, 3, 4, 5)
xj, wj = np.polynomial.hermite.hermgauss(n)
xf, wf = np.polynomial.hermite.hermgauss(60)     # regla fina para integrar

def L(j, x):
    out = np.ones_like(x)
    for k in range(n):
        if k != j:
            out *= (x - xj[k]) / (xj[j] - xj[k])
    return out

filas = []
for j in range(n):
    peso_val = np.sum(wf * L(j, xf)**2)
    peso_der = np.sum(wf * (xf - xj[j]) * L(j, xf)**2)
    filas.append([xj[j], peso_val, wj[j], peso_der])
print(pd.DataFrame(filas, columns=["nodo x_j", "∫e^{-x²}L_j²", "peso de hermgauss", "∫e^{-x²}(x-x_j)L_j²"]).to_string(index=False))

### Celda 5 · Ítem II.1 — ⟨x⁴⟩ en el estado base

**Predice en tu guía antes de correr:** ¿con cuántos nodos sale exacto ⟨x⁴⟩ = 3/4? ¿Qué da la regla con n = 2?

In [ ]:
k = 4                                            # <-- CAMBIA ESTO (potencia de x)
exacto = 0.0 if k % 2 else factorial(k) / (factorial(k // 2) * 2**k)   # ⟨x^k⟩ = (k-1)!!/2^(k/2)
filas = []
for n in range(1, 6):
    q = valor_esperado_GH(lambda x: x**k, n)
    filas.append([n, 2*n - 1, q, exacto - q])
print(f"⟨x^{k}⟩ exacto = {exacto}")
print(pd.DataFrame(filas, columns=["n", "grado máx. exacto 2n-1", "regla", "error"]).to_string(index=False))

### Celda 6 · Ítem II.2 — la corrección de Morse con n nodos

g(x) = V_M(x) − V_arm(x) = D(1 − e^{−a x})² − x²/2. La receta de primer orden (te la regalamos): **ΔE₀ ≈ ⟨ψ₀| V_M − V_arm |ψ₀⟩ = ⟨g⟩**.

Forma cerrada: como ⟨e^{−k x}⟩ = e^{k²/4} en la gaussiana, ⟨g⟩ = D(1 − 2e^{a²/4} + e^{a²}) − 1/4. La calculamos de dos maneras algebraicamente idénticas: tal cual ("ingenua") y reescrita con `expm1(t)` = eᵗ − 1 ("estable"): ⟨g⟩ = D(expm1(a²) − 2·expm1(a²/4)) − 1/4.

La columna "estimación" usa la fórmula del error de la Parte I con η = 0: D·a^{2n}(4ⁿ − 2)/(2n)! · n!/2ⁿ.

**Predice en tu guía antes de correr:** al pasar de n a n + 1, ¿el error se divide por 2, por 10, por 100…? ¿Hasta dónde puede bajar en el computador? ¿Importa contra cuál de los dos valores "exactos" comparas?

In [ ]:
g = lambda x: D * (1 - np.exp(-a * x))**2 - x**2 / 2
exacto_ingenuo = D * (1 - 2 * np.exp(a**2 / 4) + np.exp(a**2)) - 0.25
exacto_estable = D * (np.expm1(a**2) - 2 * np.expm1(a**2 / 4)) - 0.25
exacto_g = exacto_estable
print(f"⟨g⟩ ingenuo = {exacto_ingenuo:.18f} ħω")
print(f"⟨g⟩ estable = {exacto_estable:.18f} ħω   (diferencia entre los dos: {abs(exacto_ingenuo - exacto_estable):.1e})")

n_max = 12                                       # <-- CAMBIA ESTO
filas = []
for n in range(1, n_max + 1):
    q = valor_esperado_GH(g, n)
    est = D * a**(2*n) * (4**n - 2) / factorial(2*n) * factorial(n) / 2**n if n >= 2 else np.nan
    filas.append([n, q, abs(q - exacto_ingenuo), abs(q - exacto_estable), est])
tabla = pd.DataFrame(filas, columns=["n", "regla GH", "|err| vs ingenuo", "|err| vs estable", "estimación (Parte I)"])
pd.set_option("display.float_format", lambda v: f"{v:.3e}")
print(tabla.to_string(index=False))

plt.semilogy(tabla["n"], tabla["|err| vs estable"] + 1e-18, "o-", label="error vs. valor estable")
plt.semilogy(tabla["n"], tabla["|err| vs ingenuo"] + 1e-18, "x:", label="error vs. valor ingenuo")
plt.semilogy(tabla["n"], tabla["estimación (Parte I)"], "s--", label="fórmula del error (η = 0)")
plt.axhline(np.finfo(float).eps, color="gray", ls=":", label="ε_máq ≈ 2,2e-16")
plt.xlabel("número de nodos n"); plt.ylabel("|error|"); plt.legend(); plt.title("Gauss–Hermite para ⟨V_Morse − V_arm⟩")
plt.show()

### Celda 7 · Ítem II.3 — ¿cuánto se corre el nivel base, en cm⁻¹?

Taylor de Morse alrededor de x = 0 (con D·a² = 1/2): V_M(x) = x²/2 − (a/2)x³ + (7/24)a²x⁴ + …

Dos recetas que te regalamos (no tienes que demostrarlas):
- **Receta 1 (primer orden):** un término extra W(x) corre E₀ en ⟨ψ₀|W|ψ₀⟩.
- **Receta 2 (segundo orden, solo para el término cúbico):** un término b·x³ corre E₀ en −(11/8)b².

Morse tiene niveles exactos E_v = ωe(v+½) − ωe·xe(v+½)².

**Predice en tu guía antes de correr:** ¿la receta 1 sube o baja el nivel base? ¿Y la fórmula exacta de Morse? ¿Cuánto aporta el término cúbico a primer orden?

In [ ]:
# Receta 1 aplicada a toda la diferencia g = V_M − V_arm (valor de la celda 6)
dE_1 = exacto_g
# Receta 1 solo con el término cuártico (7/24) a² x⁴, usando ⟨x⁴⟩ = 3/4
dE_cuartico = 7 * a**2 / 24 * 0.75
# Receta 2 con el término cúbico b x³, b = −a/2
b_cub = -a / 2
dE_cubico = -11 / 8 * b_cub**2
incluir_cubico = True                             # <-- CAMBIA ESTO (True / False)

E0_arm = we / 2
E0_morse = we / 2 - wexe / 4
print(f"Receta 1 con toda la diferencia ⟨g⟩:  {dE_1:+.6f} ħω = {dE_1*we:+.2f} cm⁻¹")
print(f"Receta 1 solo término cuártico:       {dE_cuartico:+.6f} ħω = {dE_cuartico*we:+.2f} cm⁻¹")
if incluir_cubico:
    print(f"Receta 2, término cúbico:             {dE_cubico:+.6f} ħω = {dE_cubico*we:+.2f} cm⁻¹")
    print(f"  cuártico + cúbico:                  {dE_cuartico + dE_cubico:+.6f} ħω = {(dE_cuartico + dE_cubico)*we:+.2f} cm⁻¹")
print(f"Desplazamiento EXACTO de Morse E0(Morse) − E0(arm) = {(E0_morse - E0_arm)/we:+.6f} ħω = {E0_morse - E0_arm:+.2f} cm⁻¹")

### Celda 8 · Ítem II.4 — la frecuencia fundamental de HCl

La fundamental es la transición v = 0 → 1: ν = E₁ − E₀.
- Armónico: ν = ωe.
- Morse: ν = ωe − 2ωe·xe.
- "Observada": la tabla de NIST trae más constantes (ωe·ye, ωe·ze) ajustadas a las bandas infrarrojas medidas (hasta v = 5), con las que G(v) = ωe(v+½) − ωe·xe(v+½)² + ωe·ye(v+½)³ + ωe·ze(v+½)⁴.
  (Control independiente: la base de datos espectral del Virtual Planetary Laboratory, U. de Washington, https://vpl.astro.washington.edu/spectra/hcl.htm, lista para HCl "ν₁ 2886 cm⁻¹".)

**Predice en tu guía antes de correr:** ¿qué error relativo (%) tiene el modelo armónico? ¿Y Morse? ¿Cuántos niveles ligados tiene el pozo de Morse?

In [ ]:
G = lambda v: we*(v+0.5) - wexe*(v+0.5)**2 + weye*(v+0.5)**3 + weze*(v+0.5)**4
nu_obs   = G(1) - G(0)
nu_arm   = we
nu_morse = we - 2 * wexe
for nombre, nu in [("armónico", nu_arm), ("Morse", nu_morse)]:
    print(f"{nombre:9s}: ν = {nu:8.2f} cm⁻¹   error relativo = {100*(nu - nu_obs)/nu_obs:+.3f} %")
print(f"observada: ν = {nu_obs:8.2f} cm⁻¹   (NIST, constantes completas)")

v = np.arange(0, 6)
print(pd.DataFrame({"v": v,
                    "E_arm (cm⁻¹)": we*(v+0.5),
                    "E_Morse (cm⁻¹)": we*(v+0.5) - wexe*(v+0.5)**2,
                    "G(v) NIST (cm⁻¹)": G(v)}).to_string(index=False, float_format=lambda t: f"{t:.1f}"))
v_max = int(np.floor(2 * D - 0.5))
print(f"Niveles ligados de Morse: v = 0, 1, ..., {v_max}  ->  {v_max + 1} niveles")